In [1]:
import pandas as pd
import numpy as np
from sqlalchemy import create_engine
from dotenv import load_dotenv
import os

load_dotenv(dotenv_path="../../.env")
engine = create_engine(
    f"mysql+pymysql://{os.getenv('DB_USER')}:{os.getenv('DB_PASSWORD')}"
    f"@{os.getenv('DB_HOST')}:{os.getenv('DB_PORT')}/{os.getenv('DB_NAME')}"
)

# On ne charge QUE les colonnes utiles (pas SELECT *) -> moins de RAM
cols = [
    "`Valeur fonciere`", "`Code departement`", "`Type local`",
    "`Surface reelle bati`", "`Nombre pieces principales`", "`Surface terrain`",
    "annee", "mois", "codeRegion", "population_geo",
    "longitude", "latitude", "revenu_median", "code_commune_geo"
]
query = f"SELECT {', '.join(cols)} FROM transactions;"   # TOUTES les lignes

print("Chargement des 5,6M lignes... (1-3 min, patiente ⏳)")
df = pd.read_sql(query, con=engine)
print("Chargé :", df.shape)


Chargement des 5,6M lignes... (1-3 min, patiente ⏳)
Chargé : (5625088, 14)


In [2]:
def optimiser_memoire(df):
    # Nombres décimaux : float64 -> float32 (2x moins de RAM)
    for c in df.select_dtypes("float64").columns:
        df[c] = df[c].astype("float32")
    # Entiers : downcast automatique
    for c in df.select_dtypes("int64").columns:
        df[c] = pd.to_numeric(df[c], downcast="integer")
    # Colonnes texte répétitives -> category (énorme gain sur code_commune_geo)
    for c in ["Code departement", "Type local", "code_commune_geo"]:
        df[c] = df[c].astype("category")
    return df

avant = df.memory_usage(deep=True).sum() / 1e6
df = optimiser_memoire(df)
apres = df.memory_usage(deep=True).sum() / 1e6
print(f"Mémoire : {avant:.0f} Mo -> {apres:.0f} Mo (économie {100*(1-apres/avant):.0f}%)")


Mémoire : 1542 Mo -> 245 Mo (économie 84%)


In [3]:
from sklearn.model_selection import train_test_split

X = df.drop(columns=["Valeur fonciere"])
y = df["Valeur fonciere"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

del df, X          # on libère la mémoire des gros objets devenus inutiles
print("Train :", X_train.shape, "| Test :", X_test.shape)


Train : (4500070, 13) | Test : (1125018, 13)


In [4]:
median_terrain = X_train.loc[X_train["Type local"] == "Maison", "Surface terrain"].median()

def imputer_terrain(X, med):
    X = X.copy()
    m_app = (X["Type local"] == "Appartement") & (X["Surface terrain"].isna())
    X.loc[m_app, "Surface terrain"] = 0
    m_mai = (X["Type local"] == "Maison") & (X["Surface terrain"].isna())
    X.loc[m_mai, "Surface terrain"] = med
    return X

X_train = imputer_terrain(X_train, median_terrain)
X_test  = imputer_terrain(X_test,  median_terrain)
print("Manquants :", X_train.isna().sum().sum(), X_test.isna().sum().sum())


Manquants : 0 0


In [5]:
def ajouter_features_simples(X):
    X = X.copy()
    X["surface_totale"]    = X["Surface reelle bati"] + X["Surface terrain"]
    nb = X["Nombre pieces principales"].replace(0, np.nan)
    X["surface_par_piece"] = (X["Surface reelle bati"] / nb).fillna(0)
    X["date_num"]          = X["annee"] + (X["mois"] - 1) / 12
    X["surface_x_revenu"]  = X["Surface reelle bati"] * X["revenu_median"]
    return X

def distance_haversine(lat1, lon1, lat2, lon2):
    R = 6371
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat, dlon = lat2 - lat1, lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2)**2
    return 2 * R * np.arcsin(np.sqrt(a))

def ajouter_features_geo(X):
    X = X.copy()
    X["distance_paris"] = distance_haversine(X["latitude"], X["longitude"], 48.8566, 2.3522)
    dept = X["Code departement"].astype(str)
    X["is_paris"]           = (dept == "75").astype(int)
    X["is_petite_couronne"] = dept.isin(["75", "92", "93", "94"]).astype(int)
    return X

for f in (ajouter_features_simples, ajouter_features_geo):
    X_train, X_test = f(X_train), f(X_test)

print("Colonnes :", X_train.shape[1])


Colonnes : 20


In [6]:
from sklearn.model_selection import KFold

# --- prix_median_dept (médiane du train) ---
prix_dept = y_train.groupby(X_train["Code departement"].astype(str), observed=True).median()
gm = y_train.median()
for Xset in ("train", "test"):
    pass
X_train["prix_median_dept"] = X_train["Code departement"].astype(str).map(prix_dept).fillna(gm)
X_test["prix_median_dept"]  = X_test["Code departement"].astype(str).map(prix_dept).fillna(gm)

# --- prix_median_commune (OUT-OF-FOLD sur log) ---
def target_encode_kfold(cle_train, cible, cle_test, n_splits=5, m=20, seed=42):
    moyenne_globale = cible.mean()
    oof = pd.Series(index=cle_train.index, dtype="float32")
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=seed)
    for idx_calc, idx_enc in kf.split(cle_train):
        k = cle_train.iloc[idx_calc].astype(str)
        t = cible.iloc[idx_calc]
        agg = t.groupby(k, observed=True).agg(["mean", "count"])
        lisse = (agg["count"]*agg["mean"] + m*moyenne_globale) / (agg["count"] + m)
        oof.iloc[idx_enc] = cle_train.iloc[idx_enc].astype(str).map(lisse).fillna(moyenne_globale).values
    agg_f = cible.groupby(cle_train.astype(str), observed=True).agg(["mean", "count"])
    lisse_f = (agg_f["count"]*agg_f["mean"] + m*moyenne_globale) / (agg_f["count"] + m)
    enc_test = cle_test.astype(str).map(lisse_f).fillna(moyenne_globale)
    return oof, enc_test

X_train["prix_median_commune"], X_test["prix_median_commune"] = target_encode_kfold(
    X_train["code_commune_geo"], np.log1p(y_train), X_test["code_commune_geo"]
)

# --- encodage final : is_maison + suppression des identifiants ---
def finaliser(X):
    X = X.copy()
    X["is_maison"] = (X["Type local"] == "Maison").astype(int)
    return X.drop(columns=["Type local", "Code departement", "code_commune_geo"])

X_train = finaliser(X_train)
X_test  = finaliser(X_test)

print("Features finales :", X_train.shape[1])
print("Colonnes texte restantes ?", (X_train.dtypes == "object").any() or
      (X_train.dtypes == "category").any())


C:\Users\steph\AppData\Local\Temp\ipykernel_57320\4039123707.py:21: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[12.53077364 12.6968015  11.61491481 ... 11.82618489 12.41475015
 11.87325515]' has dtype incompatible with float32, please explicitly cast to a compatible dtype first.
  oof.iloc[idx_enc] = cle_train.iloc[idx_enc].astype(str).map(lisse).fillna(moyenne_globale).values


Features finales : 20
Colonnes texte restantes ? False


In [7]:
from lightgbm import LGBMRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

y_train_log = np.log1p(y_train)
y_test_log  = np.log1p(y_test)

modele_final = LGBMRegressor(
    n_estimators=600,        # plus d'arbres : on a beaucoup plus de données
    learning_rate=0.05,
    num_leaves=63,
    subsample=0.8,
    colsample_bytree=0.8,
    n_jobs=-1,
    random_state=42
)

print("Entraînement sur", f"{len(X_train):,}", "lignes... ⏳")
modele_final.fit(X_train, y_train_log)      # PAS de scaling : LightGBM n'en a pas besoin

# Évaluation
pred_log = modele_final.predict(X_test)
pred = np.clip(np.expm1(pred_log), 0, None)

print("\n=== Modèle final — LightGBM sur 5,6M ===")
print(f"R² (log) : {r2_score(y_test_log, pred_log):.4f}")
print(f"MAE      : {mean_absolute_error(y_test, pred):,.0f} €")
print(f"MedAE    : {np.median(np.abs(y_test.values - pred)):,.0f} €")
print(f"RMSE     : {np.sqrt(mean_squared_error(y_test, pred)):,.0f} €")


Entraînement sur 4,500,070 lignes... ⏳
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.314156 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 3007
[LightGBM] [Info] Number of data points in the train set: 4500070, number of used features: 20
[LightGBM] [Info] Start training from score 12.193162

=== Modèle final — LightGBM sur 5,6M ===
R² (log) : 0.6246
MAE      : 157,933 €
MedAE    : 41,341 €
RMSE     : 1,331,556 €


In [8]:
import joblib
from pathlib import Path

ml = Path("../../data/processed/ml")
joblib.dump(modele_final, ml / "modele_final_lightgbm.pkl")
joblib.dump(list(X_train.columns), ml / "features_order.pkl")   # utile pour l'inférence

print("✅ Modèle final sauvegardé :", ml / "modele_final_lightgbm.pkl")


✅ Modèle final sauvegardé : ..\..\data\processed\ml\modele_final_lightgbm.pkl


## 🧾 Modèle final — LightGBM sur 5,6M lignes

- R²(log) = 0,625 | MedAE ≈ 41k€ | erreur médiane ~21%.
- Entraîné sur 4,5M lignes (test 1,1M), pipeline identique au dev.
- Optimisation mémoire : 1542 → 245 Mo (float32 + category).
- Gain vs dev (560k) : +0,018 R² → confirme que ~0,62 est le plafond des DONNÉES, pas du volume.
- Pas de normalisation (LightGBM insensible à l'échelle).
